In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_pickle("df_fe_epic_big_best_customers_4c.pickle")
# filtro con date_id menor o igual a 33
df = df[df["date_id"] <= 33]
df.shape

(148260, 762)

In [3]:
# check if columns are all nan
# drop columns where all values are nan
df = df.dropna(axis=1, how='all')

In [4]:
df

,product_id,customer_id,fecha,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,plan_precios_cuidados,cust_request_qty,cust_request_tn,...,prod_tn_rolling_mean_24_x_tn_wavelet_0_mean_lag_6,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_2,prod_tn_rolling_mean_24_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_3,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_2,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_24_lag_1,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_24_lag_1_x_tn_rolling_mean_12_lag_3
0,20001,0,2017-01,NaT,NaT,NaT,NaT,NaN,425,471.382019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20001,0,2017-02,NaT,NaT,NaT,NaT,NaN,379,503.561951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20001,0,2017-03,NaT,NaT,NaT,NaT,NaN,417,756.478882,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20001,0,2017-04,NaT,NaT,NaT,NaT,NaN,209,748.621460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20001,0,2017-05,NaT,NaT,NaT,NaT,NaN,599,1069.177734,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157605,21299,0,2017-08,NaT,NaT,NaT,NaT,NaN,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157606,21299,10001,2017-08,2017-08-01,2017-08-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157607,21299,10002,2017-08,2017-08-01,2017-08-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157608,21299,10003,2017-08,2017-08-01,2017-08-01,2017-01-01,2019-12-01,0.0,1,0.005460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
    # 📄 Leer lista de productos a predecir
with open("product_id_apredecir201912.txt", "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

In [6]:
df['fecha'] = df['fecha'].apply(lambda x: x.to_timestamp('M')) 
df = df.rename(columns={'fecha': 'timestamp'})
df.drop(columns=["target"], inplace=True, errors='ignore')

In [7]:
# Filtrar hasta dic 2019 y productos requeridos
df = df[
    (df['product_id'].isin(product_ids))
]
df

,product_id,customer_id,timestamp,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,plan_precios_cuidados,cust_request_qty,cust_request_tn,...,prod_tn_rolling_mean_24_x_tn_wavelet_0_mean_lag_6,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_2,prod_tn_rolling_mean_24_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_24_x_tn_rolling_mean_12_lag_3,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_2,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_24_lag_1,prod_tn_wavelet_0_mean_lag_6_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_24_lag_1,prod_tn_rolling_mean_12_lag_2_x_tn_rolling_mean_12_lag_3,prod_tn_rolling_mean_24_lag_1_x_tn_rolling_mean_12_lag_3
0,20001,0,2017-01-31,NaT,NaT,NaT,NaT,NaN,425,471.382019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20001,0,2017-02-28,NaT,NaT,NaT,NaT,NaN,379,503.561951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20001,0,2017-03-31,NaT,NaT,NaT,NaT,NaN,417,756.478882,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20001,0,2017-04-30,NaT,NaT,NaT,NaT,NaN,209,748.621460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20001,0,2017-05-31,NaT,NaT,NaT,NaT,NaN,599,1069.177734,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157408,21276,10004,2019-06-30,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157409,21276,10004,2019-07-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157410,21276,10004,2019-08-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157411,21276,10004,2019-09-30,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
#serie id es product_id-customer_id
df['serie_id'] = df['product_id'].astype(str) + '-' + df['customer_id'].astype(str)

In [9]:
# transformo cat1, cat2, cat3, brand y sku_size a categoricos
df['cat1'] = df['cat1'].astype('category')
df['cat2'] = df['cat2'].astype('category')
df['cat3'] = df['cat3'].astype('category')
df['brand'] = df['brand'].astype('category')
df['sku_size'] = df['sku_size'].astype('category')


In [10]:
static_features_df = pd.DataFrame({
    'cat1': df.groupby('serie_id')['cat1'].first(),
    'cat2': df.groupby('serie_id')['cat2'].first(),
    'cat3': df.groupby('serie_id')['cat3'].first(),
    'brand': df.groupby('serie_id')['brand'].first(),
    'sku_size': df.groupby('serie_id')['sku_size'].first(),
    "customer_id": df.groupby('serie_id')['customer_id'].first(),
    "product_id": df.groupby('serie_id')['product_id'].first(),
}).reset_index()
static_features_df

,serie_id,cat1,cat2,cat3,brand,sku_size,customer_id,product_id
0,20001-0,HC,ROPA LAVADO,Liquido,ARIEL,3000.0,0,20001
1,20001-10001,HC,ROPA LAVADO,Liquido,ARIEL,3000.0,10001,20001
2,20001-10002,HC,ROPA LAVADO,Liquido,ARIEL,3000.0,10002,20001
3,20001-10003,HC,ROPA LAVADO,Liquido,ARIEL,3000.0,10003,20001
4,20001-10004,HC,ROPA LAVADO,Liquido,ARIEL,3000.0,10004,20001
...,...,...,...,...,...,...,...,...
3895,21276-0,PC,PIEL1,Cara,NIVEA,140.0,0,21276
3896,21276-10001,PC,PIEL1,Cara,NIVEA,140.0,10001,21276
3897,21276-10002,PC,PIEL1,Cara,NIVEA,140.0,10002,21276
3898,21276-10003,PC,PIEL1,Cara,NIVEA,140.0,10003,21276


In [11]:
# ⏰ 4. Crear TimeSeriesDataFrame
ts_data = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column='serie_id',
    timestamp_column='timestamp',
    static_features_df=static_features_df,

)
ts_data = ts_data.fill_missing_values()

In [12]:
# ⚙️ 5. Definir y entrenar predictor
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS',  # Frecuencia mensual (Month Start)
    #known_covariates_names=["timestamp_int", "timestamp.year", "timestamp.month", "timestamp.day", "timestamp.dayofweek"]
    verbosity=4
)

predictor.fit(
    ts_data, 
    num_val_windows=2, 
    hyperparameters={
        "TemporalFusionTransformer": {}
    }
)

Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250715_173142'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
GPU Count:          1
Memory Avail:       17.29 GB / 31.23 GB (55.3%)
Disk Space Avail:   774.73 GB / 914.78 GB (84.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': {'TemporalFusionTransformer': {}},
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 4}

train_data with frequency 'IRREG' has been resampled to fr

In [ ]:
# 🔮 6. Generar predicción
forecast = predictor.predict(ts_data)

In [ ]:
# Extraer predicción media y filtrar febrero 2020
forecast_mean = forecast['mean'].reset_index()
print(forecast_mean.columns)

In [ ]:
forecast_mean

In [ ]:

resultado = forecast['mean'].reset_index()
resultado = resultado[resultado['timestamp'] == '2020-02-01']
resultado

In [ ]:
# Filtrar solo febrero 2020
resultado = forecast['mean'].reset_index()
resultado = resultado[resultado['timestamp'] == '2020-02-01']

# agrego 2 columnas producto_id y customer_id basadas en serie_id
resultado['product_id'] = resultado['item_id'].apply(lambda x: x.split('-')[0])
resultado['customer_id'] = resultado['item_id'].apply(lambda x: x.split('-')[1])

#agrupo por producto_id y sumo tn
resultado = resultado.groupby('product_id').agg({'mean': 'sum'}).reset_index()
resultado.columns = ['product_id', 'tn']
resultado.to_csv("predicciones_febrero2020_fecha_v02_01-more-features-autogluon.csv", index=False)
resultado.head()

In [ ]:
resultado